# FirstLine model training and evaluation
Run the first two code cells for a fictional case with the bundled model. Set FIRSTLINE_TRAIN_CSV to your authorized FinalMlfast.csv to reproduce the selected fit and hospital holdouts.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import sys

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(name, "4")

here = Path.cwd()
MODEL_DIR = (here if (here / "model.joblib").is_file() else here / "model").resolve()
if not (MODEL_DIR / "baseline.py").is_file():
    raise FileNotFoundError("Start Jupyter in the repository root or model directory.")
sys.path.insert(0, str(MODEL_DIR))

import joblib
import numpy as np
import pandas as pd
from baseline import SOURCE_VITALS, LABELS, intake_features, make_pipeline, metrics, prepare_training_features
from cohort import cohort_mask
from experiment_features import ExperimentModel, MIN_GROUP_TRAIN, fit_classifier, routing_groups

manifest = json.loads((MODEL_DIR / "manifest.json").read_text(encoding="utf-8"))
complaints = manifest["complaint_columns"]

In [ ]:
intake = dict(age=46, gender="Male", arrival="Walk-in", hr=110, rr=24, spo2=94,
              temp=37.2, sbp=148, dbp=92)
example = intake_features(intake, ["cc_chestpain"], complaints)
saved_model = joblib.load(MODEL_DIR / "model.joblib")
p = saved_model.predict_proba(example)[0]
print("Suggested ESI for fictional case", int(LABELS[p.argmax()]))

## Repeat the selected experiment
Keep the original export outside the repository or in an ignored directory. The other model comparisons are provided in the saved results.

In [ ]:
source = os.environ.get("FIRSTLINE_TRAIN_CSV")
if source:
    source = Path(source).expanduser()
    with source.open("rb") as handle:
        digest = hashlib.file_digest(handle, "sha256").hexdigest()
    if digest != manifest["source_csv_sha256"]:
        raise ValueError("The source does not match the original training export.")
    numeric = ["esi", "age", *SOURCE_VITALS, *complaints]
    raw = pd.read_csv(source, usecols=[*numeric, "dep_name", "gender", "arrivalmode"],
                      dtype={key: "float32" for key in numeric})
    visits = raw.loc[cohort_mask(raw)]
    features = prepare_training_features(visits, complaints).reset_index(drop=True)
    target = visits["esi"].to_numpy(dtype=np.int8)
    hospitals = visits["dep_name"].to_numpy(dtype=str)
    assert len(features) == 384375 and set(hospitals) == {"A", "B", "C"}
else:
    print("Training skipped. Set FIRSTLINE_TRAIN_CSV to repeat it.")

In [ ]:
def fit_selected(features, labels):
    preprocessing = make_pipeline(complaints).named_steps["preprocessing"].fit(features)
    encoded = preprocessing.transform(features).astype(np.float32)
    shared = fit_classifier(encoded, labels, weighted=True)
    specialists = {}
    groups = routing_groups(features, complaints)["complaint"]
    for group in sorted(set(groups)):
        mask = groups == group
        if mask.sum() >= MIN_GROUP_TRAIN and len(np.unique(labels[mask])) >= 2:
            specialists[group] = fit_classifier(encoded[mask], labels[mask], weighted=True)
    return ExperimentModel(preprocessing, shared, specialists, "complaint", complaints, "none")

if source:
    probabilities = np.empty((len(target), len(LABELS)), dtype=np.float32)
    for hospital in sorted(set(hospitals)):
        held_out = hospitals == hospital
        fold = fit_selected(features.loc[~held_out], target[~held_out])
        probabilities[held_out] = fold.predict_proba(features.loc[held_out])
    predictions = LABELS[probabilities.argmax(axis=1)]
    report = metrics(target, predictions, probabilities)
    report["within_one"] = float(np.mean(np.abs(predictions - target) <= 1))
    print(json.dumps({key: report[key] for key in (
        "n", "accuracy", "within_one", "undertriage_rate", "overtriage_rate",
        "undertriage_by_two_or_more_rate"
    )}, indent=2))

In [ ]:
if source:
    artifacts = MODEL_DIR.parent / "artifacts"
    artifacts.mkdir(exist_ok=True)
    refit = fit_selected(features, target)
    joblib.dump(refit, artifacts / "model-refit.joblib", compress=3)
    (artifacts / "held-out-metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")